# ReviewIQ — Notebook 01: Amazon Reviews 2023 Dataset Ingestion
**Dataset:** `McAuley-Lab/Amazon-Reviews-2023` from HuggingFace

This notebook:
- Loads the Amazon Reviews 2023 dataset
- Selects key product categories
- Cleans and normalizes fields
- Saves processed parquet files for downstream notebooks

**Run on:** RHOAI / OpenShift AI Workbench (Python 3.10+, GPU optional for this step)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install datasets huggingface_hub pandas pyarrow tqdm -q

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm
import huggingface_hub

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# HF token - set via env var HF_TOKEN or replace below
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in to Hugging Face")
else:
    print("⚠️  No HF_TOKEN set — rate limits apply. Set env var HF_TOKEN for faster downloads.")

# Categories most relevant for ReviewIQ B2B SaaS customers
# (CPG brands, DTC sellers, Amazon private label)
CATEGORIES = [
    "Health_and_Household",
    "Beauty_and_Personal_Care",
    "Grocery_and_Gourmet_Food",
    "Sports_and_Outdoors",
    "Home_and_Kitchen",
    "Baby_Products",
    "Pet_Supplies",
    "Electronics",
]

# Max reviews per category to keep compute manageable
# Increase for full training runs
MAX_PER_CATEGORY = 100_000

print(f"Target categories: {CATEGORIES}")
print(f"Max reviews/category: {MAX_PER_CATEGORY:,}")


In [ ]:
# ── Load dataset per category ─────────────────────────────────────────────────
# Load directly from HF Hub Parquet files — bypasses the deprecated loading script
# No datasets library script execution needed

import requests
import io

HF_BASE = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

all_dfs = []

for category in tqdm(CATEGORIES, desc="Loading categories"):
    print(f"
⏳ Loading {category}...")
    try:
        # Fetch the parquet index to find available part files
        index_url = f"{HF_BASE}/raw/review/{category}/part_0.parquet"
        
        dfs = []
        part = 0
        total_rows = 0
        
        while total_rows < MAX_PER_CATEGORY:
            url = f"{HF_BASE}/raw/review/{category}/part_{part}.parquet"
            resp = requests.get(url, headers=headers, stream=True)
            
            if resp.status_code == 404:
                break  # No more parts
            
            resp.raise_for_status()
            df_part = pd.read_parquet(io.BytesIO(resp.content))
            
            remaining = MAX_PER_CATEGORY - total_rows
            if len(df_part) > remaining:
                df_part = df_part.iloc[:remaining]
            
            dfs.append(df_part)
            total_rows += len(df_part)
            part += 1
            
            if total_rows >= MAX_PER_CATEGORY:
                break
        
        if dfs:
            df = pd.concat(dfs, ignore_index=True)
            df["category"] = category
            all_dfs.append(df)
            print(f"  ✅ {len(df):,} reviews loaded from {part} part(s)")
        else:
            print(f"  ⚠️  No parts found for {category}")

    except Exception as e:
        print(f"  ❌ Failed: {e}")

if all_dfs:
    raw_df = pd.concat(all_dfs, ignore_index=True)
    print(f"
📦 Total raw reviews: {len(raw_df):,}")
    print(raw_df.dtypes)
    raw_df.head(3)
else:
    print("No data loaded — check errors above")


In [ ]:
# ── Schema inspection ─────────────────────────────────────────────────────────
print("Columns:", raw_df.columns.tolist())
print("\nData types:")
print(raw_df.dtypes)
print("\nNull counts:")
print(raw_df.isnull().sum())

In [ ]:
# ── Clean and normalize ───────────────────────────────────────────────────────
def clean_reviews(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Rename to ReviewIQ schema
    rename_map = {
        "rating": "star_rating",
        "text": "review_text",
        "title": "review_title",
        "asin": "asin",
        "user_id": "reviewer_id",
        "timestamp": "review_date",
        "verified_purchase": "verified_purchase",
        "helpful_vote": "helpful_votes",
        "images": "has_images",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # Drop rows with no review text
    df = df.dropna(subset=["review_text"])
    df = df[df["review_text"].str.strip().str.len() > 10]

    # Normalize star rating to float
    if "star_rating" in df.columns:
        df["star_rating"] = pd.to_numeric(df["star_rating"], errors="coerce")
        df = df.dropna(subset=["star_rating"])
        df["star_rating"] = df["star_rating"].clip(1, 5)

    # Binary sentiment label for classification
    # 4-5 stars = positive (1), 1-2 stars = negative (0), 3 = neutral (skip for binary)
    df["sentiment_label"] = df["star_rating"].apply(
        lambda r: 1 if r >= 4 else (0 if r <= 2 else None)
    )

    # Parse date
    if "review_date" in df.columns:
        df["review_date"] = pd.to_datetime(df["review_date"], unit="ms", errors="coerce")

    # Has images flag
    if "has_images" in df.columns:
        df["has_images"] = df["has_images"].apply(
            lambda x: bool(x) if isinstance(x, list) else bool(x)
        )

    # Helpful votes to int
    if "helpful_votes" in df.columns:
        df["helpful_votes"] = pd.to_numeric(df["helpful_votes"], errors="coerce").fillna(0).astype(int)

    # Text length feature (useful for fake review detection)
    df["review_length"] = df["review_text"].str.len()
    df["word_count"] = df["review_text"].str.split().str.len()

    # Combined text field for model input
    df["full_text"] = (
        df.get("review_title", pd.Series("", index=df.index)).fillna("") 
        + " " 
        + df["review_text"].fillna("")
    ).str.strip()

    return df


clean_df = clean_reviews(raw_df)
print(f"After cleaning: {len(clean_df):,} reviews")
print(f"Sentiment distribution:")
print(clean_df["sentiment_label"].value_counts())
clean_df.head(3)

In [ ]:
# ── Train/Val/Test split ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Use only binary-labeled rows for classification training
labeled_df = clean_df.dropna(subset=["sentiment_label"]).copy()
labeled_df["sentiment_label"] = labeled_df["sentiment_label"].astype(int)

train_df, temp_df = train_test_split(
    labeled_df, test_size=0.2, random_state=42, stratify=labeled_df["category"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["category"]
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"\nLabel distribution (train):")
print(train_df["sentiment_label"].value_counts(normalize=True).round(3))

In [ ]:
# ── Save to parquet ───────────────────────────────────────────────────────────
clean_df.to_parquet(PROCESSED_DIR / "reviews_clean_all.parquet", index=False)
train_df.to_parquet(PROCESSED_DIR / "reviews_train.parquet", index=False)
val_df.to_parquet(PROCESSED_DIR / "reviews_val.parquet", index=False)
test_df.to_parquet(PROCESSED_DIR / "reviews_test.parquet", index=False)

# Also save per-category for competitor analysis
for cat, grp in clean_df.groupby("category"):
    grp.to_parquet(PROCESSED_DIR / f"reviews_{cat}.parquet", index=False)

print("✅ All files saved to:", PROCESSED_DIR)
print("\nFiles:")
for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
# ── Dataset summary card ──────────────────────────────────────────────────────
summary = {
    "total_reviews": len(clean_df),
    "unique_asins": clean_df["asin"].nunique(),
    "categories": clean_df["category"].nunique(),
    "date_range": [
        str(clean_df["review_date"].min().date()) if "review_date" in clean_df else "N/A",
        str(clean_df["review_date"].max().date()) if "review_date" in clean_df else "N/A",
    ],
    "avg_star_rating": round(clean_df["star_rating"].mean(), 2),
    "verified_purchase_pct": round(
        clean_df["verified_purchase"].mean() * 100 if "verified_purchase" in clean_df else 0, 1
    ),
    "avg_review_length": round(clean_df["review_length"].mean(), 1),
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
}

import json
(PROCESSED_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))